# dbt: incremental models, Iceberg time travel and snapshots

Rebuilding a big table on every run does not scale. This notebook uses three dbt features that matter for lakehouse tables:

1. **Incremental models** with the `merge` strategy: only new and changed rows are processed, as one Iceberg `MERGE` (one snapshot) per run.
2. **Iceberg time travel** on the tables dbt built, from Trino and Spark.
3. **Snapshots**: dbt's slowly-changing-dimension (SCD type 2) history.

Prerequisite: none; the notebook rebuilds the demo project first.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
from lab_utils import dbt, trino, reset_dbt_demo, DBT_PROJECT

reset_dbt_demo()
dbt("build");

## 1. How `fct_orders` is built incrementally
The model (`models/marts/fct_orders.sql`) configures `materialized='incremental'`, `unique_key='order_id'`, `incremental_strategy='merge'` and, for Iceberg, `partitioning = day(order_ts)`.
On the first run dbt creates the table. On later runs the `{% if is_incremental() %}` branch selects only rows from the last day before the newest `order_ts` already loaded (a **look-back** window, so late changes are still caught),
and dbt merges them into the table on `order_id`.

In [ ]:
print((DBT_PROJECT / "models/marts/fct_orders.sql").read_text())

In [ ]:
SNAP_SQL = """SELECT snapshot_id, committed_at, operation, element_at(summary, 'added-records') AS added_records
              FROM iceberg.dbt_demo."fct_orders$snapshots" ORDER BY committed_at"""
trino(SNAP_SQL)

One snapshot so far: the initial load. Now pretend an ingestion job lands **two new orders** and a **late status change** (order 60 was returned):

In [ ]:
trino("INSERT INTO iceberg.dbt_demo_raw.raw_orders VALUES (61, 3, 2, 2, TIMESTAMP '2026-09-21 09:00:00', 'placed'), (62, 4, 1, 1, TIMESTAMP '2026-09-21 10:30:00', 'placed')")
trino("UPDATE iceberg.dbt_demo_raw.raw_orders SET status = 'returned' WHERE order_id = 60")
dbt("run", "--select", "fct_orders")

dbt reports `MERGE (n rows)`: only the look-back window was processed, not the whole table. Check the result and the new Iceberg snapshot:

In [ ]:
display(trino("SELECT order_id, status FROM iceberg.dbt_demo.fct_orders WHERE order_id IN (60, 61, 62) ORDER BY order_id"))
trino(SNAP_SQL)

The statement dbt actually sent to Trino is saved under `target/run/`. It is a plain `MERGE INTO ... USING (...)` on the unique key:

In [ ]:
lines = [l.rstrip() for l in (DBT_PROJECT / "target/run/lakehouse_demo/models/marts/fct_orders.sql").read_text().splitlines() if l.strip()]
start = next(i for i, l in enumerate(lines) if l.strip().startswith("merge into"))
print("\n".join(l[:150] for l in lines[start:start + 12]))

## 2. Time travel on a dbt-built table
Every run produced an Iceberg snapshot, so earlier states of `fct_orders` are queryable (same data files, no copies):

In [ ]:
snaps = trino(SNAP_SQL)
first, last = int(snaps.snapshot_id.iloc[0]), int(snaps.snapshot_id.iloc[-1])
before = trino(f"SELECT count(*) AS rows_at_first_snapshot FROM iceberg.dbt_demo.fct_orders FOR VERSION AS OF {first}")
now = trino(f"SELECT count(*) AS rows_now FROM iceberg.dbt_demo.fct_orders FOR VERSION AS OF {last}")
print(before.to_string(index=False)); print(now.to_string(index=False))
trino(f"SELECT order_id, status FROM iceberg.dbt_demo.fct_orders FOR VERSION AS OF {first} WHERE order_id = 60")

## 3. The same table from Spark
dbt wrote through Trino; Spark reads the same Iceberg table (through the same REST catalog) and also exposes the metadata tables, e.g. the partitions created by `partitioning = day(order_ts)`.

In [ ]:
from lab_utils import get_spark
spark = get_spark("dbt-crossengine")
spark.sql("SELECT partition, record_count, file_count FROM iceberg_catalog.dbt_demo.fct_orders.partitions ORDER BY partition").show(8, truncate=False)
spark.sql("SELECT count(*) AS rows_seen_by_spark, round(sum(amount), 2) AS revenue FROM iceberg_catalog.dbt_demo.fct_orders").show()

## 4. `--full-refresh`
Changing the model's logic or schema? `--full-refresh` rebuilds the table from scratch instead of merging. The Iceberg table is recreated, so its snapshot history starts over:

In [ ]:
dbt("run", "--select", "fct_orders", "--full-refresh")
trino(SNAP_SQL)

## 5. Why `unique_key` must really be unique: tests protect the build
`MERGE` requires each target row to match **one** source row. If the source gets a duplicate `order_id`, the merge fails with `MERGE_TARGET_ROW_MULTIPLE_MATCHES`.
`dbt build` avoids the bad merge: the `unique` test on `stg_orders` fails first, and everything downstream of it is **skipped**.

In [ ]:
trino("INSERT INTO iceberg.dbt_demo_raw.raw_orders VALUES (62, 4, 1, 1, TIMESTAMP '2026-09-21 10:30:00', 'placed')")    # duplicate of order 62
result = dbt("build", "--select", "stg_orders+")
result[result.status.isin(["error", "fail", "skipped"])][["node", "type", "status"]]

In [ ]:
trino("DELETE FROM iceberg.dbt_demo_raw.raw_orders WHERE order_id = 62")
trino("INSERT INTO iceberg.dbt_demo_raw.raw_orders VALUES (62, 4, 1, 1, TIMESTAMP '2026-09-21 10:30:00', 'placed')")    # one copy again
result = dbt("build", "--select", "stg_orders+")
result.status.value_counts()

## 6. Snapshots: history of a changing dimension (SCD type 2)
`snapshots/customers_snapshot.sql` watches the customer table with `strategy='check'` on `plan` and `country`. When a watched column changes, dbt closes the old row (`dbt_valid_to`) and inserts a new one.

In [ ]:
current = trino("SELECT plan FROM iceberg.dbt_demo_raw.raw_customers WHERE customer_id = 5").plan[0]
new_plan = "enterprise" if current != "enterprise" else "free"
print(f"customer 5: {current} -> {new_plan}")
trino(f"UPDATE iceberg.dbt_demo_raw.raw_customers SET plan = '{new_plan}' WHERE customer_id = 5")
dbt("snapshot")

In [ ]:
trino("""SELECT customer_id, plan, dbt_valid_from, dbt_valid_to FROM iceberg.dbt_demo_snapshots.customers_snapshot
         WHERE customer_id = 5 ORDER BY dbt_valid_from""")

Two rows for customer 5: the old plan with a `dbt_valid_to`, and the current plan with `dbt_valid_to = NULL`. Under the hood this is another Iceberg `MERGE`, so it is also versioned and time-travelable.

## Notes
* Incremental models on Iceberg work with `merge` (used here) and `append`; `delete+insert` is the other common strategy.
* Keep the look-back window larger than your source's lateness, and run `--full-refresh` when logic changes.
* The demo data lives in the schemas `dbt_demo`, `dbt_demo_raw` and `dbt_demo_snapshots`; `reset_dbt_demo()` drops its tables (not the schemas) for a clean start.